# Discussion and Participation Rate with Clustered SE

In [41]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')

In [42]:
%%stata
clear all
set more off
set varabbrev off

global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"

****************************************************
* Import & sample restriction
****************************************************
import delimited "$PROCESSED_DATA/proposals_panel.csv", clear varn(1) case(lower)
keep if switch_delegation == 1

****************************************************
* Mean comparison with DAO-clustered t-stat
****************************************************
tempname fh
tempfile tex1 tex2 tex3
file open `fh' using `tex1', write replace

file write `fh' "{" _n
file write `fh' "\def\sym#1{\ifmmode^{#1}\else\(^{#1}\)\fi}" _n
file write `fh' "\begin{tabular}{l*{1}{cccc}}" _n
file write `fh' "\toprule" _n
file write `fh' "                    &With Discussion&Without Discussion&With - Without&      t-stat         \\" _n
file write `fh' "\midrule" _n

local vars non_whale_participation whale_participation

foreach v of local vars {
    quietly summarize `v' if have_discussion == 1
    local mu_with = r(mean)

    quietly summarize `v' if have_discussion == 0
    local mu_without = r(mean)

    quietly regress `v' i.have_discussion, vce(cluster space)
    local diff = _b[1.have_discussion]
    local tstat = _b[1.have_discussion] / _se[1.have_discussion]
    local pval = 2 * ttail(e(df_r), abs(`tstat'))

    if (`pval' < 0.01)      local stars "***"
    else if (`pval' < 0.05) local stars "**"
    else if (`pval' < 0.10) local stars "*"
    else                    local stars ""

    if "`v'" == "non_whale_participation" local label "__SMALL_PARTICIPATION__"
    else if "`v'" == "whale_participation" local label "__WHALE_PARTICIPATION__"
    else local label "`v'"

    local mu_with_fmt    : display %9.3f `mu_with'
    local mu_without_fmt : display %9.3f `mu_without'
    local diff_fmt       : display %9.3f `diff'
    local tstat_fmt      : display %9.2f `tstat'

    file write `fh' "`label'&       `mu_with_fmt'&       `mu_without_fmt'&      `diff_fmt'&       `tstat_fmt'\sym{`stars'}\\" _n
}

file write `fh' "\bottomrule" _n
file write `fh' "\end{tabular}" _n
file write `fh' "}" _n
file close `fh'

filefilter `tex1' `tex2', from("__SMALL_PARTICIPATION__") to("$\BSmathit{Participation}^{Small}_{i,t}$") replace
filefilter `tex2' "$TABLES/sum_discussion_t.tex", from("__WHALE_PARTICIPATION__") to("$\BSmathit{Participation}^{Whale}_{i,t}$") replace



. clear all

. set more off

. set varabbrev off

. 
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. 
. ****************************************************
. * Import & sample restriction
. ****************************************************
. import delimited "$PROCESSED_DATA/proposals_panel.csv", clear varn(1) case(lo
> wer)
(encoding automatically selected: ISO-8859-1)
(64 vars, 2,830 obs)

. keep if switch_delegation == 1
(2,256 observations deleted)

. 
. ****************************************************
. * Mean comparison with DAO-clustered t-stat
. ****************************************************
. tempname fh

. tempfile tex1 tex2 tex3

. file open `fh' using `tex1', write replace
(file /var/folders/p7/57djmsb90gj7wyfdvc5r1qs80000gn/T//St10287.00002e not
    found)

. 
. file write `fh' "{" _n

. file write `fh' "\def\sym#1{\ifmmode^{#1}\else\(^{#1}\)\fi}" _n

. file write `fh' "\be